<h1>3 - Loading Data</h1>
<i>Designing loading pipelines for various kinds of file types.</i>

### Load imports and secrets

In [1]:
import os
import sys
from dotenv import load_dotenv

from openai import OpenAI
from anthropic import Anthropic

load_dotenv()

True

In [2]:
print("OPENAI Ключ найден:" if "OPENAI_API_KEY" in os.environ else "Ключ не найден")
print("ANTHROPIC Ключ найден:" if "ANTHROPIC_API_KEY" in os.environ else "Ключ не найден")
print("GOOGLE Ключ найден:" if "GOOGLE_API_KEY" in os.environ else "Ключ не найден")

OPENAI Ключ найден:
ANTHROPIC Ключ найден:
GOOGLE Ключ найден:


### 1.1 Loading Word Files in Python

Option 1: load word files using the `python_docx` library

In [4]:
import os
import requests
from docx import Document
from io import BytesIO

file_path = "../../datasets/word_files/2023_Jan_7_Feature_Engineering_Techniques.docx"
doc = Document(file_path)

text = []
for paragraph in doc.paragraphs:
    text.append(paragraph.text)

full_text = "\n".join(text)

In [7]:
print(full_text)


7 of the Most Used Feature Engineering Techniques
Hands-on Feature Engineering with Scikit-Learn, Tensorflow, Pandas and Scipy
7 of the most used Feature Engineering Techniques — Image by the author

Table of content
Introduction
1. Encoding
 1.1 Label Encoding using Scikit-learn
 1.2 One-Hot Encoding using Scikit-learn, Pandas and Tensorflow
2. Feature Hashing
 2.1 Feature Hashing using Scikit-learn
3. Binning / Bucketizing
 3.1 Bucketizing using Pandas
 3.2 Bucketizing using Tensorflow
 3.3 Bucketizing using Scikit-learn
4. Transformer
 4.1 Log-Transformer using Numpy
 4.2 Box-Cox Function using Scipy
5. Normalize / Standardize
 5.1 Normalize and Standardize using Scikit-learn
6. Feature Crossing
 6.1 Feature Crossing in Polynomial Regression
 6.2 Feature Crossing and the Kernel-Trick
7. Principal Component Analysis (PCA)
 7.1 PCA using Scikit-learn
Summary
References

Introduction
Feature engineering describes the process of formulating relevant features that describe the underlyin

Option 2: load word files using the unstructured library

In [8]:
from unstructured.partition.docx import partition_docx
import pandas as pd

elements = partition_docx(filename=file_path)

list_of_elements = []

for element in elements:
    element_dict = {
        "element_id": element.id,
        "file_path": file_path,
        "category": element.category,  # e.g. "Title", "NarrativeText", "ListItem"
        "text": element.text,
        "last_modified": element.metadata.last_modified,
    }

    list_of_elements.append(element_dict)

elements_df = pd.DataFrame(list_of_elements)

In [9]:
elements_df.head()

,element_id,file_path,category,text,last_modified
0,135f726911a68beceb56d92e2b9d10bc,../../datasets/word_files/2023_Jan_7_Feature_E...,Title,7 of the Most Used Feature Engineering Techniques,2026-05-14T19:59:12
1,f275447183f11b993f2a87d4b428299b,../../datasets/word_files/2023_Jan_7_Feature_E...,Title,Hands-on Feature Engineering with Scikit-Learn...,2026-05-14T19:59:12
2,9dacd0881e31b366756a6cc20884f661,../../datasets/word_files/2023_Jan_7_Feature_E...,NarrativeText,7 of the most used Feature Engineering Techniq...,2026-05-14T19:59:12
3,3bec63fc43107e87aae98bbaf5313196,../../datasets/word_files/2023_Jan_7_Feature_E...,Title,Table of content,2026-05-14T19:59:12
4,b1b29811f875047fef0bab817d6325c5,../../datasets/word_files/2023_Jan_7_Feature_E...,UncategorizedText,Introduction,2026-05-14T19:59:12


### 1.2 Loading PDF Files

In [ ]:
# PyPDF2 Pillow
import PyPDF2
import pandas as pd

file_path = "../../datasets/pdf_files/2023_Jan_7_Feature_Engineering_Techniques.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    # Initialize an empty string to store the extracted text
    list_of_pages = []
    page_counter = 1

    for page in reader.pages:
        page_dict = {
            "file_name": reader.metadata.get("/Title"),
            "producer": reader.metadata.get("/Producer"),
            "page_number": page_counter,
            "text": page.extract_text(),
            "images": page.images,
        }

        list_of_pages.append(page_dict)

        page_counter += 1

# Convert the list of pages to a pandas DataFrame
pages_df = pd.DataFrame(list_of_pages)

In [13]:
pages_df.head()

,file_name,producer,page_number,text,images
0,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,1,7\nof\nthe\nMost\nUsed\nFeature\nEngineering\n...,"[File(name=X7.png, data: 1.9 kB)]"
1,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,2,3.2\nBucketizing\nusing\nTensorflow\n3.3\nBuck...,[]
2,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,3,A\nstandard\nMachine\nLearning\npipeline — Ins...,"[File(name=X17.png, data: 699 Byte)]"
3,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,4,"●\nI\nn\nthe\nsupply\nchain\ncontext\n,\nevery...","[File(name=X20.png, data: 3.1 kB)]"
4,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,5,Once\nwe\nhave\nenough\ndata\nthat\ndescribes\...,"[File(name=X26.png, data: 1.7 kB)]"


### 1.3 Loading and Handling CSV and Excel Files

In [16]:
# openpyxl pandas
import pandas as pd

file_path = "../../datasets/csv_files/census-income.xlsx"
df_excel = pd.read_excel(io=file_path)


def create_text_description_of_row(row):
    row["text_description"] = (
        f"""The candidate {row['age']} years old is working in the
            {row['workclass']} sector. The candidate was born in
            {row['native-country']}, is {row['marital-status']}
            and has a {row['relationship']} relationship.
            The candidate has a {row['education']} degree
            and is working as a {row['occupation']}.
            The income of the candidate is {row['income']}."""
    )

    return row


# Apply the function create_text_description_of_row to each row of the data frame
df_extended = df_excel.apply(create_text_description_of_row, axis=1)

In [17]:
df_extended["text_description"].head()

0    The candidate 39 years old is working in the\n...
1    The candidate 50 years old is working in the\n...
2    The candidate 38 years old is working in the\n...
3    The candidate 53 years old is working in the\n...
4    The candidate 28 years old is working in the\n...
Name: text_description, dtype: str

### 1.4 Querying a PostgreSQL Database

This section needs a PostgreSQL database to connect to.

**Note:** Make sure a PostgreSQL server is installed and running, either locally on your machine or on a remote host that you can connect to.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from psycopg2 import OperationalError

# PostgreSQL connection parameters
username = "rag_user"
password = "raguserpassword123"
host = "localhost"
port = "5432"
database = "postgres"

connection_string = (
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

try:
    engine = create_engine(connection_string)
    with engine.connect() as connection:
        query = """SELECT * FROM categories ORDER BY category_id ASC """
        result = pd.read_sql(query, connection)
        print(result)
except OperationalError as e:
    print(f"Error connecting to PostgreSQL database: {e}")
    print("Please ensure the PostgreSQL server is running and accessible.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### 1.5 Loading Audio Files by Using Speech-to-Text Models

In [22]:
import os
from openai import OpenAI

audio_file_path = "../../datasets/audio_files/harvard.wav"

# initialize the OpenAI client with your API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

with open(audio_file_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1", file=audio_file
    )

transcription

Transcription(text='The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.', logprobs=None, usage=UsageDuration(seconds=19.0, type='duration'))

### 1.6 Extracting Text from Images and PDFs Using OCR

install Tesseract and its developer tools on Ubuntu \
sudo apt install tesseract-ocr\
sudo apt install libtesseract-dev\

In [24]:
# pdf2image pytesseract pillow
from pdf2image import convert_from_path
from PIL import Image
import pytesseract

image = Image.open(
    fp="../../datasets/images/example_finance_reporting_slide.png"
)

text = pytesseract.image_to_string(image)

In [26]:
print(text)

Sales volume growth driven by EMEA 18/19

Business Development
1,926

628 e Sales volume growth driven by

EMEA with sales activity up to 2.3x
YoY

e Sales activity in 20 countries during
Q2 19
e US (28%), Spain (20%), India
(15%) and Norway (11%) are the
main contributors to the Q2 19
sales volume

~
Ww
x
—
®
£
2
i}
>
7)
2
©
7)

Q4 18 Q119

Americas TT EMEA




If you have a scanned PDF, you can load it and convert each page into an image, and
then use Tesseract to extract the text from each page:\

sudo apt install poppler-utils

In [28]:
from pdf2image import convert_from_path
from PIL import Image
import pytesseract

file_path = (
    "../../datasets/pdf_files/2023_Jan_7_Feature_Engineering_Techniques.pdf"
)

images = convert_from_path(pdf_path=file_path)

text = []
for i, image in enumerate(images):
    page_text = pytesseract.image_to_string(image)
    text.append(page_text)

print(text)

['7 of the Most Used Feature Engineering Techniques\n\nHands-on Feature Engineering with Scikit-Learn, Tensorflow, Pandas and Scipy\n\n7 of the most used Feature Engineering Techniques—Image by the author\n\nTable of content\n\nIntroduction\n\n1. Encoding\n\n1.1 Label Encoding using Scikit-learn\n\n1.2 One-Hot Encoding using Scikit-learn, Pandas and Tensorflow\n2. Feature Hashing\n\n2.1 Feature Hashing using Scikit-learn\n\n3. Binning / Bucketizing\n\n3.1 Bucketizing using Pandas\n', '3.2 Bucketizing using Tensorflow\n\n3.3 Bucketizing using Scikit-learn\n\n4. Transformer\n\n4.1 Log-Transformer using Numpy\n\n4.2 Box-Cox Function using Scipy\n\n5. Normalize / Standardize\n\n5.1 Normalize and Standardize using Scikit-learn\n6. Feature Crossing\n\n6.1 Feature Crossing in Polynomial Regression\n6.2 Feature Crossing and the Kernel-Trick\n\n7. Principal Component Analysis (PCA)\n\n7.1 PCA using Scikit-learn\n\nSummary\n\nReferences\n\nIntroduction\n\nFeature engineering describes the proces

Newer OCR engines (EasyOCR, PaddleOCR)\
Improved accuracy on handwriting and complex layouts while maintaining local
deployment benefits

### 1.7 Extracting Text from Images using Multimodal Models

In [32]:
import base64
from openai import OpenAI

png_file_path = "../../datasets/images/example_finance_reporting_slide.png"

# initialize the OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

with open(png_file_path, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    prompt = (
        "Extract the text from the image attached. Make sure to only "
        "extract only the text. If there is no text in the image, "
        "please return with the sentence 'No text found in the image."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # define the model to use
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                f"data:image/jpeg;base64,"
                                f"{base64_image}"
                            ),
                        },
                    },
                ],
            }
        ],
        max_completion_tokens=500,
    )

    content = response.choices[0].message.content
    print(content)

Sales volume growth driven by EMEA 18/19

Business Development

• Sales volume growth driven by EMEA with sales activity up to 2.3x YoY

• Sales activity in 20 countries during Q2 19
  • US (28%), Spain (20%), India (15%) and Norway (11%) are the main contributors to the Q2 19 sales volume


### 1.8 Generating Text Summaries for Images Using Multimodal Models

In [33]:
import base64
from openai import OpenAI

image_path = "../../datasets/images/vietnam.png"

# initialize the OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

with open(image_path, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    prompt = (
        "You are an assistant for visually impaired users. "
        "Describe the image in detail."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                f"data:image/jpeg;base64,"
                                f"{base64_image}"
                            ),
                        },
                    },
                ],
            }
        ],
        max_completion_tokens=150,
    )

    content = response.choices[0].message.content
    print(content)

The image depicts a vibrant city skyline at dusk, showcasing a mix of modern architecture and beautiful waterfront views. 

On the left side, there is a uniquely shaped skyscraper with a pointed top, which stands out prominently among the other buildings. Next to it, multiple glass high-rises with a variety of designs gleam with lights as night begins to fall. The buildings display a range of colors, reflecting the city’s dynamic energy.

The waterfront is calm and features a gentle ripple effect where the water reflects the colors of the sky and the illuminated structures. In the foreground, there appears to be a dock or pier with a boat, adding a sense of movement to the scene. 

The sky showcases a gradient effect, transitioning from soft blues to pur


### 1.9 Generating Text Summaries for Embedded Tables Using Multimodal Models

In [ ]:
from unstructured.partition.pdf import partition_pdf

pdf_file_path = "../../datasets/pdf_files/adult_data_article.pdf"

tables = []
texts = []

# partition the PDF file into its elements
raw_pdf_elements = partition_pdf(
    filename=pdf_file_path,
    strategy="hi_res",
)

for element in raw_pdf_elements:
    if "unstructured.documents.elements.Table" in str(type(element)):
        tables.append(str(element))

In [ ]:
from openai import OpenAI
import pandas as pd

def summarize_tables(row):
    summary_prompt = (
        f"You are an assistant tasked with summarizing tables. "
        f"Give a concise summary of the table. "
        f"Table chunk: {row.table}"
    )

    # Initialize the OpenAI API client and generate table summary
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": summary_prompt}],
        temperature=0.7,
        max_tokens=150,
    )

    row["table_summary"] = response.choices[0].message.content

    return row


# create a pandas dataframe from the tables
tables_df = pd.DataFrame(tables, columns=["table"])

# add a column to the dataframe to store the summaries
tables_df = tables_df.apply(summarize_tables, axis=1)

In [ ]:
# define a random question to the embedded table
user_question = "What are the education levels of the people working in Sales?"


def build_prompt_and_generate_answer(user_question, found_table):
    """
    This function builds a prompt using the user's question and the context of the table
    and generates an answer using the OpenAI API

    Parameters:
        user_question: the question asked by the user
        found_table: the table context to generate the answer from

    Returns:
        answered_question: the answer to the user's question
    """

    question_prompt = f"""You are an assistant using the content from PDFs \
                        to answer questions. Below you can find the \
                        user's question and relevant context. Please use the \
                        context to generate an answer to the user's question.

                        # User question: {user_question}

                        # Context:

                        ## Table summary:
                        {found_table.table_summary}

                        ## Table content:
                        {found_table.table}""".strip()

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    answered_question = (
        client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": question_prompt}],
            temperature=0.7,
            max_tokens=150,
        )
        .choices[0]
        .message.content
    )

    return answered_question


# generate the answer to the user's question
# as context we using the first entry in the tables_df
answered_question = build_prompt_and_generate_answer(
    user_question=user_question, found_table=tables_df.iloc[0]
)

print(answered_question)

The individuals working in Sales have the following education levels:

1. Some-college
2. HS-grad (High School Graduate)
3. Some-college
4. Bachelors

### 1.10 Parsing PDFs with Multiple Media Content Using Unstructured and Multimodal Models

In [ ]:
from unstructured.partition.pdf import partition_pdf
import os

# set the OCR agent to tesseract
os.environ["OCR_AGENT"] = "tesseract"

pdf_file_path = "../../datasets/pdf_files/adult_data_article.pdf"
image_output_dir = "../../datasets/extracted_content_from_pdfs/images"

# create output directory if it doesn't exist
os.makedirs(image_output_dir, exist_ok=True)

# get elements using the function extract_pdf_elements
raw_pdf_elements = partition_pdf(
    filename=pdf_file_path,
    extract_images_in_pdf=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=False,
    extract_image_block_output_dir=image_output_dir,
)

# categorize elements by type
tables = []
texts = []
titles = []

# fill the just created lists with the elements
for element in raw_pdf_elements:
    element_type = str(type(element))
    if "unstructured.documents.elements.Table" in element_type:
        tables.append(str(element))
    elif "unstructured.documents.elements.NarrativeText" in element_type:
        texts.append(str(element))
    elif "unstructured.documents.elements.Title" in element_type:
        titles.append(str(element))

### 1.11 Loading Videos Using Speech-to-Text and Multimodal Models

In [ ]:
import os
import pandas as pd

from moviepy import VideoFileClip, TextClip, CompositeVideoClip

video_file_path = "../datasets/videos/learn-data-science-tutorial.mp4"
image_output_folder = "../datasets/videos/video_extracted_images"

# create output folder if it doesn't exist
os.makedirs(image_output_folder, exist_ok=True)

# Check if the video file exists
if os.path.exists(video_file_path):
    clip = VideoFileClip(video_file_path)

    # create a list of timestamps from which to extract a frame
    time_step = 10  # time in seconds
    timestamps = list(range(0, int(clip.duration) - time_step, time_step))

    # for each timestamp extract a frame
    for timestamp in timestamps:
        frame_image_path = os.path.join(
            image_output_folder, f"frame_{timestamp}.png"
        )
        clip.save_frame(frame_image_path, t=timestamp)
else:
    print(f"Video file not found: {video_file_path}. Skipping video processing.")
    timestamps = [] # Ensure timestamps is defined even if file not found

In [ ]:
# for each timestamp extract the audio sequence and save it to a .mp3 file
audio_output_folder = "../datasets/videos/video_extracted_audio"

# create output folder if it doesn't exist
os.makedirs(audio_output_folder, exist_ok=True)

for timestamp in timestamps:
    audio_clip = clip.subclip(timestamp, timestamp + time_step).audio
    output_audio_path = os.path.join(
        audio_output_folder, f"audio_{timestamp}.mp3"
    )
    audio_clip.write_audiofile(output_audio_path)

In [ ]:
import os
from openai import OpenAI


def audio_to_text(audio_path):
    """
    Convert audio to text using OpenAI's Whisper model.
    """

    if not os.path.exists(audio_path):
        print(f"[ERROR] File does not exist: {audio_path}")
        return None

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    try:
        with open(audio_path, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file
            )

        text_file_path = os.path.splitext(audio_path)[0] + ".txt"

        with open(text_file_path, "w", encoding="utf-8") as text_file:
            text_file.write(transcription.text)

        return transcription.text

    except FileNotFoundError:
        print(f"[ERROR] File not found during processing: {audio_path}")
    except PermissionError:
        print(f"[ERROR] Permission denied: {audio_path}")
    except Exception as e:
        print(f"[ERROR] Failed to process {audio_path}: {e}")

    return None


# Process folder
audio_files = os.listdir(audio_output_folder)

for audio_file in audio_files:
    absolute_path_audio_file = os.path.join(audio_output_folder, audio_file)

    # No need to check exists here anymore, function handles it
    result = audio_to_text(audio_path=absolute_path_audio_file)

    if result is None:
        print(f"Skipping file: {absolute_path_audio_file}")